## 0. Setup
**Το τρέχουμε την πρώτη μόνο φορά! Εγκατάσταση βιβλιοθηκών.**

In [ ]:
# SETUP ENVIRONMENT & DATA DOWNLOAD
print("⏳ Installing libraries...")
%pip install -q pandas numpy scipy statsmodels seaborn matplotlib requests altair
print("✅ Libraries installed.")

## 1. Import Libraries & Setup

Σε αυτό το βήμα εισάγουμε όλες τις απαραίτητες βιβλιοθήκες για την ανάλυση δεδομένων.
* `pandas`: Για τη διαχείριση των δεδομένων (DataFrames).
* `numpy`: Για μαθηματικούς υπολογισμούς (π.χ. λογάριθμους, ρίζες).
* `scipy.stats`: Για στατιστικά τεστ (Pearson correlation, T-tests).
* `statsmodels`: Για προηγμένη στατιστική μοντελοποίηση (OLS Regression, ANOVA).
* `seaborn` & `matplotlib`: Για την οπτικοποίηση των αποτελεσμάτων (διαγράμματα).

Επίσης, ορίζουμε το στυλ των διαγραμμάτων ώστε να είναι ευανάγνωστα.

In [ ]:
import pandas as pd
import numpy as np
import scipy.stats as stats
import statsmodels.api as sm
import statsmodels.formula.api as smf
import seaborn as sns
import matplotlib.pyplot as plt
from statsmodels.stats.anova import anova_lm
import os
import shutil
import altair as alt
from IPython.display import display, HTML
import matplotlib.lines as mlines
import warnings


sns.set_theme(style="whitegrid")
%matplotlib inline

## 2. Correlation Between Age and Gender in GPT-2

Στόχος είναι να εξετάσουμε τη συσχέτιση μεταξύ των διαστάσεων ηλικίας και φύλου στο μοντέλο GPT-2. 
Φορτώνουμε το αρχείο `GPT2-large-dimensions.csv` και υπολογίζουμε τον συντελεστή συσχέτισης Pearson ($r$).

Επιπλέον, υπολογίζουμε το 95% Διάστημα Εμπιστοσύνης (Confidence Interval - CI) χρησιμοποιώντας τον μετασχηματισμό Fisher (Fisher transformation), καθώς η κατανομή του $r$ δεν είναι κανονική για υψηλές συσχετίσεις. Τέλος, παρουσιάζουμε τα αποτελέσματα σε έναν πίνακα (DataFrame).

In [ ]:
# Φόρτωση δεδομένων
df_gpt = pd.read_csv('GPT2-large-dimensions.csv')

df_gpt.columns = [c.replace('.', '_') for c in df_gpt.columns]

df_gpt.rename(columns={'Social_Category': 'category'}, inplace=True)

# Pearson correlation
r, p = stats.pearsonr(df_gpt['age_norm_main'], df_gpt['gender_norm_main'])
n = len(df_gpt)

# 95% Confidence Interval
z = np.arctanh(r)
se = 1 / np.sqrt(n - 3)
z_ci = z + np.array([-1.96, 1.96]) * se
r_ci = np.tanh(z_ci)

# DataFrame αποτελεσμάτων
results_df = pd.DataFrame({
    'n': [n],
    'r': [r],
    'CI95%': [f"[{r_ci[0]:.2f}, {r_ci[1]:.2f}]"],
    'p-val': [p],
    'BF10': ['inf'], 
    'power': [1.0]
}, index=['pearson'])

display(results_df)

## 3. Robustness Check using Heatmaps

Για να επιβεβαιώσουμε ότι τα αποτελέσματα είναι ισχυρά (robust), εξετάζουμε τις συσχετίσεις μεταξύ διαφορετικών μεθόδων εξαγωγής των διαστάσεων ηλικίας και φύλου.
Δημιουργούμε δύο Heatmaps:
1.  Ένα για τις διαστάσεις της **Ηλικίας** (Age dimensions).
2.  Ένα για τις διαστάσεις του **Φύλου** (Gender dimensions).

Υψηλές συσχετίσεις (κόκκινο χρώμα) υποδηλώνουν ότι οι διαφορετικές μέθοδοι μέτρησης συμφωνούν μεταξύ τους.

In [ ]:
# στήλες df_gpt
heatmap_cols = [
    'age_main', 'age_ext', 'age_red',
    'gender_main', 'gender_ext', 'gender_red'
]

# Δημιουργία df_heat
df_heat = df_gpt[heatmap_cols].copy()

df_heat.rename(columns={
    'age_main': 'age_score',
    'gender_main': 'gender_score'
}, inplace=True)

# μεταβλητές για άξονες
age_vars = ['age_red', 'age_ext', 'age_score']
gender_vars = ['gender_red', 'gender_ext', 'gender_score']
combined_vars = age_vars + gender_vars

# Δημιουργία ticks
ticks_6x6 = np.arange(0.2, 1.01, 0.1)
ticks_3x3 = np.arange(0.2, 1.01, 0.05)

# Συνάρτηση για κόψιμο Colorbar
def crop_colorbar(ax, min_val, max_val, ticks_list):
    cbar = ax.collections[0].colorbar
    valid_ticks = [t for t in ticks_list if t >= min_val and t <= max_val]
    cbar.set_ticks(valid_ticks)
    cbar.ax.set_ylim(min_val, max_val)
    cbar.outline.set_visible(False)

# age (3x3)
plt.figure(figsize=(6, 5))
ax1 = sns.heatmap(df_heat[age_vars].corr(), annot=True, cmap='coolwarm',
            vmin=-1, vmax=1, fmt='.2f', linewidths=.5,
            cbar_kws={'ticks': ticks_3x3})
crop_colorbar(ax1, 0.59, 1.0, ticks_3x3)
plt.title('Correlation Heatmap - Age')
plt.show()

# gender (3x3)
plt.figure(figsize=(6, 5))
ax2 = sns.heatmap(df_heat[gender_vars].corr(), annot=True, cmap='coolwarm',
            vmin=-1, vmax=1, fmt='.2f', linewidths=.5,
            cbar_kws={'ticks': ticks_3x3})
crop_colorbar(ax2, 0.74, 1.0, ticks_3x3)
plt.title('Correlation Heatmap - Gender')
plt.show()

# combined (6x6)
plt.figure(figsize=(9, 7))
ax3 = sns.heatmap(df_heat[combined_vars].corr(), annot=True, cmap='coolwarm',
            vmin=-1, vmax=1, fmt='.2f', linewidths=.5,
            cbar_kws={'ticks': ticks_6x6})
crop_colorbar(ax3, 0.19, 1.0, ticks_6x6)
plt.title('Combined Correlation Heatmap (Age & Gender)')
plt.show()

## 4. Relationship between Age and Gender (OLS Regression)

Εκτελούμε μια γραμμική παλινδρόμηση (OLS) για να μοντελοποιήσουμε τη σχέση: Age - Gender
Συγκεκριμένα, χρησιμοποιούμε τη `age_norm_main` ως εξαρτημένη μεταβλητή και τη `gender_norm_main` ως ανεξάρτητη.

Στη συνέχεια, δημιουργούμε ένα διάγραμμα διασποράς (scatter plot) με τη γραμμή παλινδρόμησης. Επισημαίνουμε με ετικέτες (annotations) μερικά χαρακτηριστικά σημεία (outliers) για να δούμε ποια επαγγέλματα/λέξεις αποκλίνουν περισσότερο.

In [ ]:
blue_words = [
    "chairman of the board", "elected official", "director of research", 
    "chief of staff", "military personnel"
]
orange_words = [
    "homoeopath", "intern", "cook", "novice", "secretary"
]

def get_color(category):
    if category in blue_words: return 'Male/Old bias'
    elif category in orange_words: return 'Female/Young bias'
    else: return 'Other'

df_gpt['highlight_group'] = df_gpt['category'].apply(get_color)

model_gpt = smf.ols("age_norm_main ~ gender_norm_main", data=df_gpt)
results_gpt = model_gpt.fit()

# css για να προσομοιασουμε το παραδειγμα
summary_html = results_gpt.summary().as_html()
style = """
<style>
    .simple-frame {
        border: 2px solid white !important;
        background-color: black;
        color: white;
        padding: 10px;
        display: inline-block;
        font-family: monospace;
    }
    .simple-frame table { border-collapse: collapse; border: none !important; }
    .simple-frame td, .simple-frame th { border: none !important; padding: 5px 10px; text-align: right; }
</style>
"""
display(HTML(style + f"<div class='simple-frame'>{summary_html}</div>"))

# υπολογισμος θεσης

offsets = {
    # ΜΠΛΕ
    "chairman of the board": {'dx': -0.15, 'dy': 0.15, 'align': 'left'},
    "elected official":      {'dx': -0.15, 'dy': 0.15, 'align': 'right'},
    "director of research":  {'dx': 0.05, 'dy': -0.05, 'align': 'left'},
    "chief of staff":        {'dx': 0.05, 'dy': -0.05, 'align': 'left'},
    "military personnel":    {'dx': -0.25, 'dy': 0.05, 'align': 'right'},

    # ΠΟΡΤΟΚΑΛΙ
    "homoeopath":            {'dx': -0.15, 'dy': 0.05, 'align': 'right'},
    "intern":                {'dx': -0.10, 'dy': -0.05, 'align': 'right'},
    "cook":                  {'dx': 0.05,  'dy': -0.15, 'align': 'left'},
    "novice":                {'dx': 0.20,  'dy': 0.00,  'align': 'left'},
    "secretary":             {'dx': 0.10,  'dy': -0.20, 'align': 'left'}
}

labels_df = df_gpt[df_gpt['highlight_group'] != 'Other'].copy()

def calculate_pos(row, axis):
    params = offsets.get(row['category'], {'dx': 0.05, 'dy': 0.04})
    if axis == 'x':
        return row['gender_norm_main'] + params['dx']
    if axis == 'y':
        return row['age_norm_main'] + params['dy']

labels_df['label_x'] = labels_df.apply(lambda r: calculate_pos(r, 'x'), axis=1)
labels_df['label_y'] = labels_df.apply(lambda r: calculate_pos(r, 'y'), axis=1)
labels_df['align'] = labels_df['category'].apply(lambda x: offsets.get(x, {'align': 'left'})['align'])

labels_left = labels_df[labels_df['align'] == 'left']
labels_right = labels_df[labels_df['align'] == 'right']

# Grid Dataframes
x_grid = pd.DataFrame({'x': np.arange(-0.25, 1.35, 0.25)})
y_grid = pd.DataFrame({'y': np.arange(-0.2, 1.25, 0.2)})

# Regression Line Dataframe
intercept = results_gpt.params['Intercept']
slope = results_gpt.params['gender_norm_main']

line_df = pd.DataFrame({'x': [-0.2, 1.4]}) 
line_df['y'] = intercept + slope * line_df['x']

# Σχεδίαση με Altair

base = alt.Chart(df_gpt).encode(
    x=alt.X('gender_norm_main', 
            title=['Gender Association', '(Female-Male Dimension)'],
            scale=alt.Scale(domain=[-0.2, 1.3]),
            axis=alt.Axis(values=[0, 0.5, 1.0],
                          titleFontWeight='bold', titleFontSize=14,
                          grid=False)), 
    y=alt.Y('age_norm_main', 
            title=['Age Association', '(Young-Old Dimension)'],
            scale=alt.Scale(domain=[-0.20, 1.15]),
            axis=alt.Axis(values=[0.0, 0.4, 0.8],
                          titleFontWeight='bold', titleFontSize=14,
                          grid=False))
)

# Layer 1: Grid
grid_x_layer = alt.Chart(x_grid).mark_rule(color='white', strokeWidth=1).encode(x='x')
grid_y_layer = alt.Chart(y_grid).mark_rule(color='white', strokeWidth=1).encode(y='y')

# Layer 2: Κόκκινη Γραμμή
reg_line_layer = alt.Chart(line_df).mark_line(color='red', size=2).encode(x='x', y='y')

# Layer 3: Γκρι Σημεία
grey_points = base.mark_point(
    color='black', 
    fill='transparent', 
    opacity=0.5, 
    size=20 
).transform_filter(alt.datum.highlight_group == 'Other')

# Layer 4: Γραμμές Σύνδεσης
lines_layer = alt.Chart(labels_df).mark_rule(color='black', opacity=0.5, strokeWidth=0.5).encode(
    x='gender_norm_main',
    y='age_norm_main',
    x2='label_x',
    y2='label_y'
)

# Layer 5: Μαρκαρισμένα Σημεία
highlighted_points = base.mark_circle(
    size=80, 
    opacity=1, 
    stroke='black', 
    strokeWidth=1
).encode(
    color=alt.Color('highlight_group', 
                    scale=alt.Scale(domain=['Male/Old bias', 'Female/Young bias'],
                                    range=['#1f77b4', '#e6a532']), legend=None)
).transform_filter(alt.datum.highlight_group != 'Other')

# Layer 6: Κείμενα
text_left = alt.Chart(labels_left).mark_text(
    align='left', 
    baseline='middle', 
    dx=3, 
    fontSize=13,
    fontWeight='bold'
).encode(
    x='label_x', y='label_y', text='category', color=alt.value('black')
)

text_right = alt.Chart(labels_right).mark_text(
    align='right', 
    baseline='middle', 
    dx=-3, 
    fontSize=13,
    fontWeight='bold'
).encode(
    x='label_x', y='label_y', text='category', color=alt.value('black')
)

# Σύνθεση
final_chart = (
    grid_x_layer + grid_y_layer + 
    reg_line_layer + 
    lines_layer + 
    grey_points + 
    highlighted_points + 
    text_left + text_right
).properties(
    width=800, height=700
).configure_view(
    fill='#EBEBEB',
    stroke=None
).configure_title(
    fontSize=16, fontWeight='bold', anchor='middle'
)

final_chart

In [ ]:
blue_words = [
    "chairman of the board", "elected official", "director of research",
    "chief of staff", "military personnel"
]

orange_words = [
    "homoeopath", "intern", "cook", "novice", "secretary"
]

def get_color(category):
    if category in blue_words:
        return 'Male/Old bias'
    elif category in orange_words:
        return 'Female/Young bias'
    else:
        return 'Other'

df_gpt['highlight_group'] = df_gpt['category'].apply(get_color)


#Διαδραστικό Γράφημα
axis_step = [0, 0.1, 0.2, 0.3, 0.4, 0.5, 0.6, 0.7, 0.8, 0.9, 1.0]

base = alt.Chart(df_gpt).encode(
    # ΑΞΟΝΑΣ X
    x=alt.X('gender_norm_main',
            title=['Gender Association', '(Female-Male Dimension)'],
            scale=alt.Scale(domain=[0, 1]),
            axis=alt.Axis(
                titleFontWeight='bold',
                titleFontSize=14,
                values=axis_step
            )
    ),
            
    # ΑΞΟΝΑΣ Y
    y=alt.Y('age_norm_main',
            title=['Age Association', '(Young-Old Dimension)'],
            scale=alt.Scale(domain=[0, 1]),
            axis=alt.Axis(
                titleFontWeight='bold',
                titleFontSize=14,
                values=axis_step
            )
    )
)

# Layer 1: Γκρι τελείες
grey_points = base.mark_circle(size=60, opacity=0.6, color="#c7c4c4", stroke='black', strokeWidth=0.5).encode(
    tooltip=['category', 'gender_norm_main', 'age_norm_main']
).transform_filter(
    alt.datum.highlight_group == 'Other'
)

# Layer 2: Χρωματιστές τελείες
highlighted_points = base.mark_circle(size=90, opacity=1, stroke='black', strokeWidth=1).encode(
    color=alt.Color('highlight_group', 
                    scale=alt.Scale(domain=['Male/Old bias', 'Female/Young bias'],
                                    range=["#62b2ff", '#e6a532']), 
                    legend=None),
    tooltip=['category', 'gender_norm_main', 'age_norm_main']
).transform_filter(
    alt.datum.highlight_group != 'Other'
)

# Layer 3: Κόκκινη γραμμή
reg_line = base.transform_regression(
    'gender_norm_main', 'age_norm_main'
).mark_line(color='red', size=2)

# Συνδυασμός
final_chart = (grey_points + reg_line + highlighted_points).properties(
    width=800,
    height=700
).configure_title(
    fontSize=16,
    fontWeight='bold',
    anchor='middle'
)

final_chart.save('age_gender_regression_interactive.html')
final_chart

## Ερμηνεία των (Outliers)
Η λεπτομερής εξέταση της οπτικοποίησης αποκαλύπτει μια σειρά από όρους που, παρόλο που αποτελούν σημαντικές αποκλίσεις από τη γραμμή παλινδρόμησης, δεν αναλύθηκαν διεξοδικά στην αρχική μελέτη. Όροι όπως "brother-in-law", "widow", "lass", "colonel", "producer", "critic", "hero", "neighbor" και "jew" καταλαμβάνουν κομβικές θέσεις στον χάρτη συσχετίσεων.

Η απόφαση των αναλυτών να μην σχολιάσουν αυτά τα αποτελέσματα μπορεί να ερμηνευτεί μέσα από τρεις κύριους άξονες:

***1. Σημασιολογικοί Περιορισμοί (Definitional Constraints)***
Πολλές από τις εξαιρέσεις που εντοπίσαμε εμπίπτουν στην κατηγορία των όρων με προκαθορισμένη γλωσσική σημασία.

* **Όροι Συγγένειας:** Λέξεις όπως "brother-in-law", "mother-in-law" και "widow" (χήρα) φέρουν εγγενώς πληροφορία για το βιολογικό φύλο και, συχνά, για την ηλικιακή ωριμότητα.

* **Ηλικιακοί Προσδιορισμοί:** Η λέξη "lass" (κορίτσι) ορίζει από μόνη της μια νέα γυναίκα.

* **Συμπέρασμα:** Οι ερευνητές πιθανώς θεώρησαν ότι η ικανότητα του μοντέλου να τοποθετεί σωστά αυτούς τους όρους αποτελεί «γνώση λεξικού» και όχι κοινωνική προκατάληψη. Αντίθετα, η μελέτη εστίασε σε «ουδέτερα» επαγγέλματα, όπου η μεροληψία είναι κοινωνικά κατασκευασμένη και όχι γλωσσικά επιβεβλημένη.

***2. Εστίαση στο Επαγγελματικό Στερεότυπο (Occupational Bias)***
Ο κεντρικός σκοπός της έρευνας ήταν η ανάδειξη των προκαταλήψεων αποκλειστικά στον εργασιακό χώρο.

* **Κοινωνικοί Ρόλοι:** Όροι που αφορούν κοινωνικές σχέσεις ή ταυτότητες, όπως "neighbor" (γείτονας), "hero" (ήρωας) ή "jew" (εβραίος), εκφεύγουν από το αυστηρό πλαίσιο της επαγγελματικής απασχόλησης.

* **Στρατηγική Επιλογή:** Οι αναλυτές επέλεξαν να αγνοήσουν αυτούς τους όρους για να διατηρήσουν τη θεματική καθαρότητα της έρευνας, αποφεύγοντας τον «θόρυβο» από κατηγορίες που δεν σχετίζονται με την αγορά εργασίας.

***3. Αντιπροσωπευτικότητα και Οπτική Σαφήνεια***
Στη στατιστική οπτικοποίηση, η επισήμανση κάθε ακραίας τιμής μπορεί να καταστήσει το γράφημα δυσανάγνωστο.

* **Συνοπτική Παρουσίαση:** Ο όρος "colonel" (συνταγματάρχης) είναι ένα εξαιρετικό παράδειγμα ανδρικής/ηλικιωμένης μεροληψίας, αλλά οι αναλυτές χρησιμοποίησαν τον ευρύτερο όρο "military personnel" ως αντιπρόσωπο ολόκληρου του κλάδου.

* **Σύγχρονοι Ρόλοι:** Επαγγέλματα όπως "producer" ή "critic" εμφανίζονται ως έντονα ανδρικοί και νεανικοί ρόλοι στο GPT-2. Παρόλο που αποτελούν ενδιαφέροντα ευρήματα, ίσως κρίθηκαν λιγότερο «εμβληματικά» για την παρουσίαση της συστηματικής ενίσχυσης (amplification) των στερεοτύπων σε σχέση με όρους όπως το "chairman of the board".

**Τελικό Συμπέρασμα Ανάλυσης**
Η δική μας διερεύνηση των outliers, συμπεριλαμβανομένων των "brother-in-law" και "producer", αποδεικνύει ότι η μεροληψία του GPT-2 δεν περιορίζεται μόνο σε μεμονωμένα επαγγελματικά στερεότυπα. Αντιθέτως, το μοντέλο φαίνεται να έχει οικοδομήσει έναν ολιστικό κοινωνικό χάρτη, όπου κάθε ιδιότητα —συγγενική, επαγγελματική ή κοινωνική— είναι άρρηκτα συνδεδεμένη με συγκεκριμένες προσδοκίες φύλου και ηλικίας. Η παράλειψη αυτών των δεδομένων από τους αρχικούς ερευνητές εξυπηρετεί τη στενότερη εστίαση της μελέτης, ωστόσο η ύπαρξή τους υπογραμμίζει ότι το πρόβλημα του αλγοριθμικού bias είναι πολύ πιο εκτεταμένο και βαθιά ριζωμένο στη σημασιολογία του μοντέλου από όσο αρχικά παρουσιάστηκε.

## 5. Amplification via Google Search (Experimental Data)

Εδώ αναλύουμε τα αποτελέσματα του πειράματος που διεξήγαγαν οι ερευνητές (Treatment vs Control).
1.  Φορτώνουμε τα αρχεία `experiment_control.csv` και `experiment_treatment.csv`.
2.  Υπολογίζουμε τη **μέση ηλικία** που εκτίμησε το Control group για κάθε επάγγελμα (`category`).
3.  Συγκρίνουμε τις εκτιμήσεις του Treatment group με τον μέσο όρο του Control, δημιουργώντας τη μεταβλητή `age_diff` (Διαφορά Ηλικίας).
4.  Οπτικοποιούμε την κατανομή των διαφορών (`age_diff`) χωριστά για όσους ανέβασαν εικόνα Άνδρα (Male) και Γυναίκας (Female).

In [ ]:
df_control = pd.read_csv('experiment_control.csv')
df_treatment = pd.read_csv('experiment_treatment.csv')

# Ονομασία συνθηκών
df_control['condition'] = 'Control'
df_treatment['condition'] = 'Image'

# μέση ηλικία Control ανά Category
control_means = df_control.groupby('category')['age'].mean().reset_index()
control_means.rename(columns={'age': 'control_mean_age'}, inplace=True)

# Merge και Διαφορά
df_treatment_diff = pd.merge(df_treatment, control_means, on='category', how='left')
df_treatment_diff['age_diff'] = df_treatment_diff['age'] - df_treatment_diff['control_mean_age']

# Μέση Τιμη ια τις Κάθετες Γραμμές
mean_male = df_treatment_diff[df_treatment_diff['gender']=='Male']['age_diff'].mean()
mean_female = df_treatment_diff[df_treatment_diff['gender']=='Female']['age_diff'].mean()

# Σχεδίαση Γραφήματος (Styling)
sns.set_style("whitegrid")
plt.figure(figsize=(10, 6))
color_female = '#FFC20A'
color_male = '#0C7BDC'

sns.kdeplot(
    data=df_treatment_diff[df_treatment_diff['gender']=='Female'],
    x='age_diff',
    fill=True,
    color=color_female,
    alpha=0.5,
    linewidth=0,
    label='_nolegend_'
)

sns.kdeplot(
    data=df_treatment_diff[df_treatment_diff['gender']=='Male'],
    x='age_diff',
    fill=True,
    color=color_male,
    alpha=0.5,
    linewidth=0,
    label='_nolegend_'
)

sns.kdeplot(
    data=df_treatment_diff[df_treatment_diff['gender']=='Female'],
    x='age_diff',
    color='black',
    linewidth=1.5,
    label='_nolegend_'
)
sns.kdeplot(
    data=df_treatment_diff[df_treatment_diff['gender']=='Male'],
    x='age_diff',
    color='black',
    linewidth=1.5,
    label='_nolegend_'
)
# Κάθετες γραμμές
plt.axvline(x=0, color='black', linestyle=':', linewidth=2, label='_nolegend_')
plt.axvline(x=mean_female, color=color_female, linestyle='-', linewidth=2.5, label='_nolegend_')
plt.axvline(x=mean_male, color=color_male, linestyle='-', linewidth=2.5, label='_nolegend_')

# Όρια άξονα Χ
plt.xlim(-25, 35)

plt.xlabel('Estimated Age Relative to Control', fontsize=12)
plt.ylabel('Density', fontsize=12)

# υπομνημα
legend_female = mlines.Line2D([], [], color=color_female, marker='s', linestyle='None',
                          markersize=10, label='Female')
legend_male = mlines.Line2D([], [], color=color_male, marker='s', linestyle='None',
                          markersize=10, label='Male')

plt.legend(handles=[legend_female, legend_male],
           title='Image Uploaded',
           loc='center right',
           frameon=False,
           bbox_to_anchor=(1, 0.5))

plt.tight_layout()
plt.show()

## 6. Statistical Significance (T-tests)

Επαληθεύουμε τους ισχυρισμούς των συγγραφέων πραγματοποιώντας T-tests:
1.  **Woman vs Man Image:** Συγκρίνουμε την εκτιμώμενη ηλικία μεταξύ όσων είδαν γυναικεία εικόνα και όσων είδαν ανδρική.
2.  **Woman Image vs Control:** Συγκρίνουμε όσους είδαν γυναικεία εικόνα με την ομάδα ελέγχου.
3.  **Man Image vs Control:** Συγκρίνουμε όσους είδαν ανδρική εικόνα με την ομάδα ελέγχου.

In [ ]:
# καθαρizω dataframes
df_control = pd.read_csv('experiment_control.csv')
df_treatment = pd.read_csv('experiment_treatment.csv')

# group by category
control_means = df_control.groupby('category')['age'].mean().reset_index()
control_means.rename(columns={'age': 'control_mean_age'}, inplace=True)
# merge
df_treatment = df_treatment.merge(control_means, on='category', how='left')
# διαφορά
df_treatment['age_diff'] = df_treatment['age'] - df_treatment['control_mean_age']

# διαχωρισμός φύλων
female_data = df_treatment[df_treatment['gender'] == 'Female']
male_data = df_treatment[df_treatment['gender'] == 'Male']

print("--- Final Replication Results (Clean Run) ---\n")

# Welch's t-test man vs woman
t1, p1 = stats.ttest_ind(female_data['age'], male_data['age'], equal_var=False)
mean_diff_1 = female_data['age'].mean() - male_data['age'].mean()

p1_formatted = "< 2.2e-16" if p1 < 2.2e-16 else f"{p1:.2e}"

print(f"1. Woman vs Man (Raw Ages):")
print(f"   Diff = {mean_diff_1:.2f} years (Paper says 5.46)")
print(f"   t = {t1:.2f} (Paper says -19.07)")
print(f"   p = {p1_formatted}\n")

# One-sample t-test Woman vs Control
t2, p2 = stats.ttest_1samp(female_data['age_diff'], 0)
mean_diff_2 = female_data['age_diff'].mean()

p2_formatted = "< 2.2e-16" if p2 < 2.2e-16 else f"{p2:.2e}"

print(f"2. Woman vs Control (Residuals vs 0):")
print(f"   Diff = {mean_diff_2:.2f} years (Paper says -1.75)")
print(f"   t = {t2:.2f} (Paper says -11.32)")
print(f"   p = {p2_formatted}\n")

# Man vs Control
t3, p3 = stats.ttest_1samp(male_data['age_diff'], 0)
mean_diff_3 = male_data['age_diff'].mean()

print(f"3. Man vs Control (Residuals vs 0):")
print(f"   Diff = {mean_diff_3:.2f} years (Paper says 0.64)")
print(f"   t = {t3:.2f} (Paper says 3.42)")
print(f"   p = {p3:.4f}")

## 7. Investigate Amplification (Combined Regression)

Ενώνουμε τα δεδομένα (Control και Treatment) σε ένα ενιαίο DataFrame. 
Τρέχουμε ένα μοντέλο παλινδρόμησης με αλληλεπίδραση (interaction) για να δούμε πώς η συνθήκη (`condition`) και το φύλο (`gender`) επηρεάζουν την ηλικία (`age`).

Το μοντέλο είναι: `age ~ condition * gender + category + subj`
Χρησιμοποιούμε **Treatment coding** με:
* Reference Condition: **Control**
* Reference Gender: **Male**

In [ ]:
df_control = pd.read_csv('experiment_control.csv')
df_treatment = pd.read_csv('experiment_treatment.csv')

df_control['condition'] = 'Control'
df_treatment['condition'] = 'Image'

# Ενοποίηση
df_exp = pd.concat([df_control, df_treatment], ignore_index=True)

# 'Male' και 'Female'
df_exp = df_exp[df_exp['gender'].isin(['Male', 'Female'])]

# Εκτέλεση μοντέλου
model_amp = smf.ols("age ~ C(condition, Treatment(reference='Control')) * C(gender, Treatment(reference='Male')) + category + subj", data=df_exp)
res_amp = model_amp.fit()

# Πάνω μέρος σύνοψης
print(res_amp.summary().tables[0])

all_rows = res_amp.params.index

results_data = {
    'coef': res_amp.params[all_rows],
    'std err': res_amp.bse[all_rows],
    't': res_amp.tvalues[all_rows],
    'P>|t|': res_amp.pvalues[all_rows],
    '[0.025': res_amp.conf_int().loc[all_rows][0],
    '0.975]': res_amp.conf_int().loc[all_rows][1]
}

# Δημιουργία DataFrame
summary_table = pd.DataFrame(results_data)

# Στρογγυλοποίηση coef στα 4, τα υπόλοιπα στα 3
summary_table['coef'] = summary_table['coef'].round(4)
cols_to_round_3 = ['std err', 't', 'P>|t|', '[0.025', '0.975]']
summary_table[cols_to_round_3] = summary_table[cols_to_round_3].round(3)

print("\n")
display(summary_table)

## 8. Predictions and Residuals Analysis

Για να απομονώσουμε την επίδραση του φύλου και της συνθήκης από τη δυσκολία της κάθε κατηγορίας ή την κρίση του κάθε υποκειμένου:
1.  Τρέχουμε ένα "βασικό" μοντέλο μόνο με `category` και `subj`.
2.  Υπολογίζουμε τα **residuals**, δηλαδή τη διαφορά μεταξύ της πραγματικής ηλικίας και αυτής που προβλέπει το βασικό μοντέλο.
3.  Φτιάχνουμε δύο γραφήματα **Bar plots**:
    * Ένα με την προβλεπόμενη ηλικία ανά ομάδα.
    * Ένα με τα **residuals** ανά ομάδα.

Τα residuals μας δείχνουν την "καθαρή" επίδραση της μεροληψίας (bias), αφαιρώντας τον θόρυβο.

In [ ]:
warnings.filterwarnings('ignore')

#Μοντέλο και Προβλέψεις
model_base = smf.ols("age ~ category + subj", data=df_exp)
res_base = model_base.fit()

df_exp['pred_age'] = res_base.predict(df_exp)
df_exp['residuals'] = df_exp['age'] - df_exp['pred_age']

x_order = ['Control', 'Image']
hue_order = ['Female', 'Male'] 
custom_colors = {'Female': '#F4D03F', 'Male': '#1f77b4'} 

fig, axes = plt.subplots(1, 2, figsize=(14, 6))

# Age Predictions
sns.pointplot(
    ax=axes[0],
    x='condition',
    y='pred_age',
    hue='gender',
    data=df_exp,
    order=x_order,
    hue_order=hue_order,
    palette=custom_colors,
    dodge=0.2,
    capsize=0.1,
    errorbar='ci',
    
    linestyle='-',
    err_kws={'linewidth': 1.2},
    linewidth=1.2,
    markersize=6,
    markers=['o', 'o']
)

axes[0].set_ylabel('predicted')
axes[0].set_xlabel('Condition')
axes[0].grid(False)
# Υπόμνημα
axes[0].legend(
    loc='center right',
    frameon=True,
    facecolor='white',
    framealpha=1
)

# Residuals
sns.pointplot(
    ax=axes[1],
    x='condition',
    y='residuals',
    hue='gender',
    data=df_exp,
    order=x_order,
    hue_order=hue_order,
    palette=custom_colors,
    dodge=0.2,
    capsize=0.1,
    errorbar='ci',
    linestyle='-',
    err_kws={'linewidth': 1.2},
    linewidth=1.2,
    markersize=6,
    markers=['o', 'o']
)

axes[1].set_ylabel('residuals')
axes[1].set_xlabel('Condition')
axes[1].grid(False)

# Υπόμνημα
axes[1].legend(
    title='Gender',
    loc='center right',
    frameon=True,
    facecolor='white',
    framealpha=1
)

plt.tight_layout()
plt.show()

## Ερμηνεία των Αποτελεσμάτων

Η ανάλυση αυτή χρησιμοποιεί ένα neutral model (παλινδρόμηση της ηλικίας μόνο ως προς τις μεταβλητές category και subj). Στόχος είναι να δημιουργηθεί μια βάση αναφοράς (baseline) που λαμβάνει υπόψη τις προσδοκίες ηλικίας ανά επάγγελμα και την υποκειμενικότητα των κριτών, αγνοώντας το φύλο και τις πειραματικές συνθήκες.

* **Προβλεπόμενη Ηλικία (Αριστερό Διάγραμμα):** Το πρώτο διάγραμμα δείχνει τις προσδοκίες ηλικίας βασισμένες αποκλειστικά στις επαγγελματικές κατηγορίες και τις τάσεις των συμμετεχόντων. Παρατηρούμε ότι οι κατηγορίες που συνδέονται με άνδρες (μπλε) ξεκινούν από υψηλότερη βάση ηλικίας σε σύγκριση με τις γυναικείες (κίτρινο). Αυτό αντανακλά τα προϋπάρχοντα κοινωνικά στερεότυπα που είναι ήδη ενσωματωμένα στους επαγγελματικούς ρόλους.

* **Ανάλυση Υπολοίπων - Το Φαινόμενο της Ενίσχυσης (Δεξί Διάγραμμα):** Το δεύτερο διάγραμμα είναι το πιο κρίσιμο για τον έλεγχο της ερευνητικής υπόθεσης. Τα υπόλοιπα (residuals) αντιπροσωπεύουν την «καθαρή μεροληψία» — δηλαδή τη διαφορά μεταξύ της πραγματικής εκτίμησης ηλικίας και της «ουδέτερης» πρόβλεψης του μοντέλου.

* **Συνθήκη Ελέγχου :** Τα υπόλοιπα είναι σχετικά κοντά στο μηδέν, που σημαίνει ότι οι εκτιμήσεις των συμμετεχόντων ευθυγραμμίζονται με την ουδέτερη βάση όταν υπάρχει μόνο κείμενο.

* **Συνθήκη Εικόνας :** Παρατηρείται μια δραματική απόκλιση. Για τις γυναίκες, τα υπόλοιπα μειώνονται σημαντικά (~ -0.9), που σημαίνει ότι τα οπτικά ερεθίσματα τις κάνουν να φαίνονται πολύ νεότερες από το αναμενόμενο. Για τους άνδρες, τα υπόλοιπα αυξάνονται (~ +1.0), που σημαίνει ότι οι εικόνες τους κάνουν να φαίνονται γηραιότεροι από τη βάση αναφοράς του επαγγέλματος.

**Συμπέρασμα:** Τα αποτελέσματα επιβεβαιώνουν ότι η Αναζήτηση Εικόνων της Google δεν αντικατοπτρίζει απλώς τις υπάρχουσες προκαταλήψεις φύλου-ηλικίας, αλλά τις ενισχύει ενεργά (amplifies). Η οπτική πληροφορία προκαλεί πολύ μεγαλύτερη στρέβλωση από το κείμενο μόνο, ενισχύοντας συστηματικά τη σύνδεση των γυναικών με τη νεότητα και των ανδρών με την επαγγελματική ωριμότητα.

## 9. ANOVA Models

Τέλος, εκτελούμε ανάλυση διακύμανσης (ANOVA) για να ελέγξουμε τη στατιστική σημαντικότητα των παραγόντων.
Χρησιμοποιούμε **Sum coding** (αντί για Treatment coding) και Type 2 ANOVA.

Τρέχουμε δύο μοντέλα:
1.  `age ~ condition * gender`
2.  `age ~ condition * gender + category + subj`

In [ ]:
# ANOVA 1
model_anova1 = smf.ols("age ~ C(condition, Sum) * C(gender, Sum)", data=df_exp)
anova1_res = anova_lm(model_anova1.fit(), typ=2)
print("--- ANOVA Model 1 ---")
display(anova1_res)

# ANOVA 2
model_anova2 = smf.ols("age ~ C(condition, Sum) * C(gender, Sum) + category + subj", data=df_exp)
anova2_res = anova_lm(model_anova2.fit(), typ=2)
print("\n--- ANOVA Model 2 ---")
display(anova2_res)

### Ερμηνεία της ANOVA

Πραγματοποιήσαμε δύο ελέγχους ANOVA για να επαληθεύσουμε την εγκυρότητα των ευρημάτων μας:

* **Μοντέλο 1** (Βασικό): Ο όρος αλληλεπίδρασης condition:gender (συνθήκη:φύλο) έχει p-value 0,055, το οποίο βρίσκεται οριακά πάνω από το τυπικό όριο στατιστικής σημαντικότητας (0,05). Αυτό υποδηλώνει ότι, χωρίς τον έλεγχο άλλων παραγόντων, ο «θόρυβος» στα δεδομένα καθιστά δυσκολότερο τον εντοπισμό της επίδρασης.

* **Μοντέλο 2** (Προσαρμοσμένο): Ωστόσο, όταν ελέγχουμε για τις μεταβλητές category (επάγγελμα) και subj (συμμετέχων), συνυπολογίζοντας δηλαδή τη δυσκολία της κάθε επαγγελματικής κατηγορίας και τις ατομικές διαφορές των συμμετεχόντων, η αλληλεπίδραση condition:gender γίνεται υψηλά στατιστικά σημαντική ($p \approx 0,007$).

**Συμπέρασμα:** Αυτό αποδεικνύει ότι το φαινόμενο της ενίσχυσης (amplification effect) είναι ισχυρό. Μόλις λάβουμε υπόψη τη διακύμανση που προκύπτει από τους διαφορετικούς τύπους εργασίας και τους μεμονωμένους συμμετέχοντες (Μοντέλο 2), επιβεβαιώνουμε στατιστικά ότι η χρήση του Google Images μεταβάλλει σημαντικά τον τρόπο με τον οποίο το φύλο επηρεάζει την αντίληψη της ηλικίας.